
# HMS Raw EEG Conv1D + BiGRU Pipeline (v1)

This notebook implements a raw-signal EEG classification pipeline for HMS using:

- Bipolar montage features (16 channels)
- Signal filtering and downsampling (200 Hz -> 100 Hz)
- Conv1D + Bidirectional GRU + attention pooling
- KL divergence on soft vote targets
- Fast sanity workflow plus optional full-fold training
- Inference and `submission.csv` generation


Train/evaluate with confidence-filtered data (`agreement >= 70%`) and a deterministic ~10% holdout confident test split (seed=42).


In [ ]:
# Runtime gate for Colab vs local VM execution.
COLAB = False
COLAB_USE_DRIVE = False
COLAB_DRIVE_ROOT = '/content/drive/MyDrive/HMS_JZ'
COLAB_CONTENT_ROOT = '/content/hms'
CONFIDENT = False

print('COLAB mode:', COLAB)
print('COLAB_USE_DRIVE:', COLAB_USE_DRIVE)
print('COLAB_DRIVE_ROOT:', COLAB_DRIVE_ROOT)
print('COLAB_CONTENT_ROOT:', COLAB_CONTENT_ROOT)
print('CONFIDENT:', CONFIDENT)


In [ ]:
# Core imports, runtime bootstrap helpers, and environment checks.
import os
import json
import gc
import sys
import math
import copy
import random
import shutil
import zipfile
import warnings
import subprocess
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from scipy.signal import butter, sosfiltfilt, iirnotch, filtfilt, resample_poly
from sklearn.model_selection import StratifiedGroupKFold

# Torch check with explicit setup guidance.
try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import Dataset, DataLoader
except Exception as exc:
    raise RuntimeError(
        'PyTorch is not available in this kernel.\n'
        'Install and restart kernel, for example:\n'
        '  pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121\n'
        f'Original import error: {exc}'
    )


# Import shared pipeline utilities with path fallbacks.
_candidate_dirs = [
    Path.cwd(),
    Path('/home/littl/ECE247A_Final_Project/JZ'),
    Path('/content/drive/MyDrive/HMS_JZ'),
    Path('/content'),
]
for _d in _candidate_dirs:
    if (_d / 'conv1d_bigru_utils.py').exists() and str(_d) not in sys.path:
        sys.path.insert(0, str(_d))

# Import shared pipeline utilities.
# If you edit conv1d_bigru_utils.py during a live session, restart the kernel
# to guarantee refreshed imports.
from conv1d_bigru_utils import (
    setup_runtime,
    set_seed,
    ensure_dirs,
    build_bipolar,
    bandpass_filter,
    notch_filter,
    resample_signal,
    extract_50s_window_by_offset,
    preprocess_row_to_array,
    row_cache_key,
    cache_split_dir,
    prepare_cache,
    apply_time_shift,
    NPZEEGDataset,
    build_dataloaders,
    ConvBlock1D,
    ConvBiGRUAttention,
    build_confident_subset,
    cap_rows_per_eeg,
    compute_global_prior,
    kl_divergence_np,
    compute_prior_baseline_kl,
    train_one_epoch,
    validate,
    predict,
    run_training,
    plot_history,
)


In [ ]:

RUNTIME_SETTINGS = setup_runtime(
    COLAB,
    colab_use_drive=COLAB_USE_DRIVE,
    colab_drive_root=COLAB_DRIVE_ROOT,
    colab_content_root=COLAB_CONTENT_ROOT,
    confident=CONFIDENT,
    confident_threshold=0.70,
    confident_test_fraction=0.10,
    confident_seed=42,
)

print('Python executable:', os.sys.executable)
print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print('Runtime settings:')
for k, v in RUNTIME_SETTINGS.items():
    print(f'  {k}: {v}')
print('Train CSV path:', RUNTIME_SETTINGS['train_csv_path'])
print('Test CSV path:', RUNTIME_SETTINGS['test_csv_path'])

sanity_history = None
full_history = None
stage2_history = None


In [ ]:
set_seed(42)

In [ ]:
# Central configuration — t19: multi-scale depthwise temporal conv front-end.
# Architecture hypothesis: the t9-t18 series suggests the bottleneck may be in
# pre-GRU temporal feature encoding across timescales rather than recurrent depth,
# label smoothing, or augmentation. This run tests that hypothesis by replacing
# the single-scale Conv1D stack with an inception-style multi-scale block using
# depthwise-separable large-kernel branches (corrects the t11 implementation mistake).
# Regularization and augmentation are held at t17 values (proven best generalization).
class CFG:
    # Paths (derived from COLAB runtime settings)
    BASE_PATH = Path(RUNTIME_SETTINGS['base_path'])
    WORK_DIR = Path(RUNTIME_SETTINGS['work_dir'])
    CACHE_DIR = Path(RUNTIME_SETTINGS['cache_dir'])
    MODELS_DIR = Path(RUNTIME_SETTINGS['models_dir'])
    RESULTS_DIR = Path(RUNTIME_SETTINGS['results_dir'])
    PLOTS_DIR = Path(RUNTIME_SETTINGS['plots_dir'])

    # Data
    n_folds = 5
    fold = 0
    src_sample_rate = 200
    target_sample_rate = 100
    num_bipolar_channels = 16
    window_seconds = 50
    num_classes = 6
    class_names = ['Seizure', 'LPD', 'GPD', 'LRDA', 'GRDA', 'Other']
    vote_cols = ['seizure_vote', 'lpd_vote', 'gpd_vote', 'lrda_vote', 'grda_vote', 'other_vote']

    # Preprocessing
    bandpass_low_hz = 0.5
    bandpass_high_hz = 40.0
    bandpass_order = 4
    apply_notch = False
    notch_freq_hz = 60.0
    notch_q = 30.0

    # Sanity check / full training toggles
    sanity_mode = True
    sanity_train_size = 2000
    sanity_valid_size = 500
    sanity_epochs = 3
    full_epochs = 20

    # Two-stage full training
    use_two_stage = False
    stage1_patience = 5
    stage2_patience = 3
    stage2_conf_threshold = 0.70
    stage2_extra_epochs = 5
    stage2_lr = 1e-4

    # Run A controls
    run_a_enable_dedup = True
    run_a_max_rows_per_eeg = 4
    run_a_dedup_rule = 'top_total_votes'
    run_a_use_vote_weighting = True
    run_a_vote_weight_mode = 'sqrt_norm'

    # Team holdout leakage guard
    use_team_holdout_exclusion = True
    team_holdout_csv = 'confident_test.csv'

    # Model settings — t19: multi-scale depthwise temporal conv.
    # conv_kernels is kept for bookkeeping but ignored when use_multiscale_conv=True.
    conv_channels = [48, 96, 144, 192]
    conv_kernels = [7, 5, 5, 3]
    conv_strides = [2, 2, 2, 2]
    gru_hidden = 160
    gru_layers = 1
    dropout = 0.45  # t17 value — held fixed to isolate arch change

    # Multi-scale conv settings (t19).
    use_multiscale_conv = True
    multiscale_kernels = (3, 15, 31)  # small / medium / large; large uses depthwise-separable

    # Optimization settings
    batch_size = 32
    lr = 1e-3
    weight_decay = 5e-3  # t17 value — held fixed
    grad_clip = 1.0
    early_stopping_patience = 4

    # Target smoothing
    label_smoothing = 0.02

    # DataLoader (Colab defaults to lower workers for stability)
    num_workers = 0
    pin_memory = True

    # Mixed precision
    use_amp = torch.cuda.is_available()
    use_compile = False # NOTE: MUST BE FALSE

    # Caching
    force_rebuild_cache = False

    # Augmentation (train) — reverted to t17 values; t18 showed alpha=0.70 hurts
    use_mixup = True
    mixup_prob = 0.60
    mixup_alpha = 0.40  # t17 value; t18's 0.70 degraded both val and holdout
    time_mask_prob = 0.35
    time_mask_frac_min = 0.03
    time_mask_frac_max = 0.10
    channel_drop_prob = 0.25
    channel_drop_max = 2

    # Sample-level augmentation magnitudes
    aug_noise_std_min = 0.0015
    aug_noise_std_max = 0.02
    aug_scale_min = 0.8
    aug_scale_max = 1.2
    aug_max_shift_seconds = 0.25
    left_right_flip_prob = 0.25

    # Toggles
    run_tiny_overfit_check = False
    run_full_after_sanity = True
    prepare_test_cache_now = True

# Ensure output directories exist.
ensure_dirs([CFG.WORK_DIR, CFG.CACHE_DIR, CFG.MODELS_DIR, CFG.RESULTS_DIR, CFG.PLOTS_DIR])

print('Base path:', CFG.BASE_PATH)
print('Work dir:', CFG.WORK_DIR)
print('Cache dir:', CFG.CACHE_DIR)
print('Models dir:', CFG.MODELS_DIR)
print('Results dir:', CFG.RESULTS_DIR)
print('Sanity mode:', CFG.sanity_mode)
print('DataLoader num_workers:', CFG.num_workers)
print('Default early stopping patience:', CFG.early_stopping_patience)
print('Two-stage enabled:', CFG.use_two_stage)
print('Stage1 patience:', CFG.stage1_patience)
print('Stage2 patience:', CFG.stage2_patience)
print('Two-stage conf threshold:', CFG.stage2_conf_threshold)
print('Two-stage extra epochs:', CFG.stage2_extra_epochs)
print('Two-stage stage2 LR:', CFG.stage2_lr)
print('Run A dedup enabled:', CFG.run_a_enable_dedup, '| max rows/eeg=', CFG.run_a_max_rows_per_eeg, '| rule=', CFG.run_a_dedup_rule)
print('Run A vote weighting:', CFG.run_a_use_vote_weighting, '| mode=', CFG.run_a_vote_weight_mode)
print('Team holdout exclusion:', CFG.use_team_holdout_exclusion)
print('Team holdout CSV:', CFG.team_holdout_csv)
print('GRU layers:', CFG.gru_layers, '| GRU hidden:', CFG.gru_hidden)
print('Multi-scale conv:', CFG.use_multiscale_conv, '| kernels:', CFG.multiscale_kernels)
print('Label smoothing:', CFG.label_smoothing)
print('Dropout:', CFG.dropout, '| Weight decay:', CFG.weight_decay)
print('Aug settings: mixup', CFG.use_mixup, '| p=', CFG.mixup_prob, '| alpha=', CFG.mixup_alpha)
print('Aug settings: time_mask p=', CFG.time_mask_prob, '| frac=', (CFG.time_mask_frac_min, CFG.time_mask_frac_max))
print('Aug settings: channel_drop p=', CFG.channel_drop_prob, '| max=', CFG.channel_drop_max)
print('Aug settings: sample noise=', (CFG.aug_noise_std_min, CFG.aug_noise_std_max), '| scale=', (CFG.aug_scale_min, CFG.aug_scale_max), '| max_shift_s=', CFG.aug_max_shift_seconds)
print('Aug settings: left_right_flip_prob=', CFG.left_right_flip_prob)




The source EEG is sampled at 200 Hz, which yields 10,000 points for each 50-second window.
For rapid iteration on a T4, this notebook downsamples to 100 Hz (5,000 points) after filtering.
This preserves clinically relevant low-frequency structure while reducing memory and training time.


In [ ]:
# Metadata loading and soft-label preparation.

# Colab/local safety checks before reading metadata.
required_items = [
    CFG.BASE_PATH / 'train.csv',
    CFG.BASE_PATH / 'sample_submission.csv',
    CFG.BASE_PATH / 'train_eegs',
]
if not CONFIDENT:
    required_items.extend([
        CFG.BASE_PATH / 'test.csv',
        CFG.BASE_PATH / 'test_eegs',
    ])
missing_items = [str(p) for p in required_items if not p.exists()]
if missing_items:
    raise FileNotFoundError(f'Required data artifacts are missing under {CFG.BASE_PATH}: {missing_items}')

# Disk warning for low free space at the data mount.
_disk_total, _disk_used, _disk_free = shutil.disk_usage(CFG.BASE_PATH)
_disk_free_gb = _disk_free / (1024 ** 3)
print(f'Free disk space at {CFG.BASE_PATH}: {_disk_free_gb:.2f} GB')
if _disk_free_gb < 20:
    warnings.warn('Low free disk space (<20 GB). Caching and extraction may fail.')

name2label = {name: i for i, name in enumerate(CFG.class_names)}
label2name = {i: name for i, name in enumerate(CFG.class_names)}

train_csv_path = Path(RUNTIME_SETTINGS['train_csv_path'])
test_csv_path = Path(RUNTIME_SETTINGS['test_csv_path'])

print('Train metadata source:', train_csv_path)
print('Test metadata source:', test_csv_path)

train_df = pd.read_csv(train_csv_path)
test_df = pd.read_csv(test_csv_path)
sample_sub = pd.read_csv(CFG.BASE_PATH / 'sample_submission.csv')

TEAM_HOLDOUT_PATIENTS = set()
if getattr(CFG, 'use_team_holdout_exclusion', False):
    holdout_csv = CFG.BASE_PATH / str(CFG.team_holdout_csv)
    if not holdout_csv.exists():
        raise FileNotFoundError(f'Team holdout CSV is missing: {holdout_csv}')

    holdout_df = pd.read_csv(holdout_csv)
    if 'patient_id' not in holdout_df.columns:
        raise ValueError(f'Team holdout CSV must include patient_id: {holdout_csv}')

    TEAM_HOLDOUT_PATIENTS = set(holdout_df['patient_id'].dropna().astype(int).tolist())
    before_rows = int(len(train_df))
    train_df = train_df[~train_df['patient_id'].isin(TEAM_HOLDOUT_PATIENTS)].reset_index(drop=True)
    removed_rows = before_rows - int(len(train_df))
    overlap_after = len(set(train_df['patient_id']).intersection(TEAM_HOLDOUT_PATIENTS))

    print(f'Team holdout exclusion enabled via: {holdout_csv}')
    print(f'Excluded patients: {len(TEAM_HOLDOUT_PATIENTS)} | removed train rows: {removed_rows}')
    print('Train/holdout patient overlap after exclusion:', overlap_after)
    assert overlap_after == 0, 'Leakage: train_df overlaps team holdout patients.'

train_df['eeg_path'] = train_df['eeg_id'].apply(lambda x: str(CFG.BASE_PATH / 'train_eegs' / f'{x}.parquet'))
if CONFIDENT:
    test_df['eeg_path'] = test_df['eeg_id'].apply(lambda x: str(CFG.BASE_PATH / 'train_eegs' / f'{x}.parquet'))
else:
    test_df['eeg_path'] = test_df['eeg_id'].apply(lambda x: str(CFG.BASE_PATH / 'test_eegs' / f'{x}.parquet'))

train_df['class_label'] = train_df['expert_consensus'].map(name2label)

votes = train_df[CFG.vote_cols].values.astype(np.float32)
vote_sums = votes.sum(axis=1, keepdims=True)
vote_sums_safe = np.clip(vote_sums, 1.0, None)
soft_labels = votes / vote_sums_safe

# Optional label smoothing on target distributions.
if CFG.label_smoothing > 0:
    eps = float(CFG.label_smoothing)
    soft_labels = (1.0 - eps) * soft_labels + (eps / CFG.num_classes)
    soft_labels = soft_labels / np.clip(soft_labels.sum(axis=1, keepdims=True), 1e-8, None)

train_df['soft_labels'] = list(soft_labels)
train_df['total_votes'] = vote_sums.squeeze(-1).astype(np.float32)


if CONFIDENT:
    missing_test_vote_cols = [c for c in CFG.vote_cols if c not in test_df.columns]
    if missing_test_vote_cols:
        raise ValueError(f'Confident test CSV is missing vote columns: {missing_test_vote_cols}')

    test_df['class_label'] = test_df['expert_consensus'].map(name2label)
    test_votes = test_df[CFG.vote_cols].values.astype(np.float32)
    test_vote_sums = test_votes.sum(axis=1, keepdims=True)
    test_vote_sums_safe = np.clip(test_vote_sums, 1.0, None)
    test_soft_labels = test_votes / test_vote_sums_safe
    test_df['soft_labels'] = list(test_soft_labels)
    test_df['total_votes'] = test_vote_sums.squeeze(-1).astype(np.float32)
    print('Confident holdout test rows:', len(test_df))
    print('Confident holdout unique patients:', test_df['patient_id'].nunique() if 'patient_id' in test_df.columns else 'N/A')

# Core validation checks for label integrity.
label_sums = soft_labels.sum(axis=1)
print('Train rows:', len(train_df))
print('Train unique EEG:', train_df['eeg_id'].nunique())
print('Train unique patients:', train_df['patient_id'].nunique())
print('Test rows:', len(test_df))
print('Label sum min/max:', float(label_sums.min()), float(label_sums.max()))
print('NaN in labels:', bool(np.isnan(soft_labels).any()))
print('Total votes min/median/max:', float(train_df['total_votes'].min()), float(train_df['total_votes'].median()), float(train_df['total_votes'].max()))
print('Label smoothing:', CFG.label_smoothing)
print('Consensus distribution:')
print(train_df['expert_consensus'].value_counts())


In [ ]:
# Patient-grouped, class-stratified fold split.
sgkf = StratifiedGroupKFold(n_splits=CFG.n_folds, shuffle=True, random_state=42)
train_df = train_df.reset_index(drop=True).copy()
train_df['fold'] = -1

for fold, (_, valid_idx) in enumerate(
    sgkf.split(train_df, y=train_df['class_label'], groups=train_df['patient_id'])
):
    train_df.loc[valid_idx, 'fold'] = fold

train_fold_df = train_df[train_df['fold'] != CFG.fold].reset_index(drop=True)
valid_fold_df = train_df[train_df['fold'] == CFG.fold].reset_index(drop=True)

# Split correctness check.
overlap = set(train_fold_df['patient_id']).intersection(set(valid_fold_df['patient_id']))
print('Fold used for validation:', CFG.fold)
print('Train fold rows:', len(train_fold_df))
print('Valid fold rows:', len(valid_fold_df))
print('Patient overlap count:', len(overlap))

if 'TEAM_HOLDOUT_PATIENTS' in globals() and len(TEAM_HOLDOUT_PATIENTS) > 0:
    train_holdout_overlap = len(set(train_fold_df['patient_id']).intersection(TEAM_HOLDOUT_PATIENTS))
    valid_holdout_overlap = len(set(valid_fold_df['patient_id']).intersection(TEAM_HOLDOUT_PATIENTS))
    print('Train vs team-holdout patient overlap:', train_holdout_overlap)
    print('Valid vs team-holdout patient overlap:', valid_holdout_overlap)
    assert train_holdout_overlap == 0, 'Leakage: train fold overlaps team holdout patients.'
    assert valid_holdout_overlap == 0, 'Leakage: valid fold overlaps team holdout patients.'


In [ ]:
# Bipolar montage definition: 4 chains x 4 pairs = 16 channels.
BIPOLAR_MONTAGE = {
    'LL': [('Fp1', 'F7'), ('F7', 'T3'), ('T3', 'T5'), ('T5', 'O1')],
    'RL': [('Fp2', 'F8'), ('F8', 'T4'), ('T4', 'T6'), ('T6', 'O2')],
    'LP': [('Fp1', 'F3'), ('F3', 'C3'), ('C3', 'P3'), ('P3', 'O1')],
    'RP': [('Fp2', 'F4'), ('F4', 'C4'), ('C4', 'P4'), ('P4', 'O2')],
}

CHAIN_ORDER = ['LL', 'RL', 'LP', 'RP']
BIPOLAR_PAIRS: List[Tuple[str, str]] = []
for chain in CHAIN_ORDER:
    BIPOLAR_PAIRS.extend(BIPOLAR_MONTAGE[chain])

print('Total bipolar pairs:', len(BIPOLAR_PAIRS))
print(BIPOLAR_PAIRS)


By default, this notebook trains on a small subset first so you can quickly decide whether the model direction is promising.


In [ ]:
# Select data scope for fast sanity mode or full fold mode.
if CFG.sanity_mode:
    train_selected = train_fold_df.sample(n=min(CFG.sanity_train_size, len(train_fold_df)), random_state=42).reset_index(drop=True)
    valid_selected = valid_fold_df.sample(n=min(CFG.sanity_valid_size, len(valid_fold_df)), random_state=42).reset_index(drop=True)
    run_epochs = CFG.sanity_epochs
else:
    train_selected = train_fold_df.reset_index(drop=True)
    valid_selected = valid_fold_df.reset_index(drop=True)
    run_epochs = CFG.full_epochs

print('Selected train rows:', len(train_selected))
print('Selected valid rows:', len(valid_selected))
print('Epochs for current run:', run_epochs)

In [ ]:
# Build caches for selected train/valid and optional test.
train_cache_files = prepare_cache(train_selected, split='train', cfg=CFG, force_rebuild=CFG.force_rebuild_cache)
valid_cache_files = prepare_cache(valid_selected, split='valid', cfg=CFG, force_rebuild=CFG.force_rebuild_cache)

if CFG.prepare_test_cache_now or CONFIDENT:
    test_cache_files = prepare_cache(test_df, split='test', cfg=CFG, force_rebuild=CFG.force_rebuild_cache)
else:
    test_cache_files = []


In [ ]:
# Build DataLoaders for the currently selected run scope.
train_loader, valid_loader = build_dataloaders(train_selected, valid_selected, CFG)

print('Train batches:', len(train_loader))
print('Valid batches:', len(valid_loader))

# Data integrity test from one cached sample.
sample_fp = train_cache_files[0]
with np.load(sample_fp) as z:
    sample_x = z['x']
print('Cached sample shape:', sample_x.shape)
assert sample_x.shape == (CFG.num_bipolar_channels, CFG.window_seconds * CFG.target_sample_rate)

In [ ]:
# Split correctness assertion.
train_patients = set(train_selected['patient_id'])
valid_patients = set(valid_selected['patient_id'])
assert len(train_patients.intersection(valid_patients)) == 0, 'Patient leakage detected in split.'

# Forward/loss sanity checks.
model_probe = ConvBiGRUAttention(CFG)
xb, yb, vb = next(iter(train_loader))
logits_probe = model_probe(xb)
assert logits_probe.shape == (xb.shape[0], CFG.num_classes), 'Model output shape mismatch.'

log_probs_probe = F.log_softmax(logits_probe, dim=1)
loss_probe = F.kl_div(log_probs_probe, yb, reduction='batchmean')
assert torch.isfinite(loss_probe), 'KL loss is not finite.'
print('Probe output shape:', logits_probe.shape)
print('Probe loss:', float(loss_probe))

# Baseline gate metric.
prior_probs = compute_global_prior(train_selected)
baseline_kl = compute_prior_baseline_kl(valid_selected, prior_probs)
print('Validation KL baseline (global prior):', baseline_kl)

if CFG.run_tiny_overfit_check:
    tiny_x = xb[:32]
    tiny_y = yb[:32]
    tiny_model = ConvBiGRUAttention(CFG)
    tiny_opt = torch.optim.AdamW(tiny_model.parameters(), lr=1e-3)
    tiny_model.train()
    for step in range(50):
        tiny_opt.zero_grad()
        tiny_logits = tiny_model(tiny_x)
        tiny_loss = F.kl_div(F.log_softmax(tiny_logits, dim=1), tiny_y, reduction='batchmean')
        tiny_loss.backward()
        tiny_opt.step()
    print('Tiny overfit final loss:', float(tiny_loss.detach().cpu()))

In [ ]:
# Phase A: fast sanity training run.
sanity_history = run_training(
    train_loader=train_loader,
    valid_loader=valid_loader,
    cfg=CFG,
    baseline_kl=baseline_kl,
    epochs= 3, #run_epochs,
    checkpoint_name=f'best_conv1d_bigru_fold{CFG.fold}_sanity_t19_multiscale_dw.pt',
)


In [ ]:
# Phase B: full training run — t19 multi-scale depthwise temporal conv.
print('device name:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
full_history = None
stage2_history = None

# Separate checkpoint names per stage — prevents stage2 from overwriting stage1 artifacts.
stage1_ckpt = f'best_conv1d_bigru_fold{CFG.fold}_full_curriculum_t19_multiscale_dw_stage1.pt'
stage2_ckpt = f'best_conv1d_bigru_fold{CFG.fold}_full_curriculum_t19_multiscale_dw_stage2.pt'
stage1_ckpt_path = CFG.MODELS_DIR / stage1_ckpt
stage1_best_ckpt_path = stage1_ckpt_path.parent / (stage1_ckpt_path.stem + '_best' + stage1_ckpt_path.suffix)

if CFG.run_full_after_sanity:
    full_baseline_kl = compute_prior_baseline_kl(valid_fold_df, compute_global_prior(train_fold_df))

    stage1_train_rows_before = int(len(train_fold_df))
    stage1_train_df_run = train_fold_df
    if getattr(CFG, 'run_a_enable_dedup', False):
        stage1_train_df_run = cap_rows_per_eeg(
            train_fold_df,
            max_rows_per_eeg=CFG.run_a_max_rows_per_eeg,
            rule=CFG.run_a_dedup_rule,
        )
    stage1_train_rows_after = int(len(stage1_train_df_run))
    print(
        f'Stage1 train rows: {stage1_train_rows_before} -> {stage1_train_rows_after} '
        f'(dedup={getattr(CFG, "run_a_enable_dedup", False)})'
    )

    full_train_loader, full_valid_loader = build_dataloaders(stage1_train_df_run, valid_fold_df, CFG)
    full_history = run_training(
        train_loader=full_train_loader,
        valid_loader=full_valid_loader,
        cfg=CFG,
        baseline_kl=full_baseline_kl,
        epochs=CFG.full_epochs,
        checkpoint_name=stage1_ckpt,
        early_stopping_patience=CFG.stage1_patience,
        stage_name='stage1',
        stage_metadata={
            'stage_label': 'full_train',
            'train_rows': stage1_train_rows_after,
            'train_rows_before_dedup': stage1_train_rows_before,
            'train_rows_after_dedup': stage1_train_rows_after,
            'valid_rows': int(len(valid_fold_df)),
            'confident_threshold': None,
            'dedup_enabled': bool(getattr(CFG, 'run_a_enable_dedup', False)),
            'dedup_max_rows_per_eeg': int(getattr(CFG, 'run_a_max_rows_per_eeg', 2)),
            'dedup_rule': str(getattr(CFG, 'run_a_dedup_rule', 'top_total_votes')),
            'vote_weighting_enabled': bool(getattr(CFG, 'run_a_use_vote_weighting', False)),
            'vote_weighting_mode': str(getattr(CFG, 'run_a_vote_weight_mode', 'sqrt_norm')),
        },
    )

    if CFG.use_two_stage and CFG.stage2_extra_epochs > 0:
        stage2_train_df_raw = build_confident_subset(
            train_fold_df,
            vote_cols=CFG.vote_cols,
            agreement_threshold=CFG.stage2_conf_threshold,
        )
        stage2_train_rows_before = int(len(stage2_train_df_raw))
        stage2_train_df = stage2_train_df_raw
        if getattr(CFG, 'run_a_enable_dedup', False):
            stage2_train_df = cap_rows_per_eeg(
                stage2_train_df_raw,
                max_rows_per_eeg=CFG.run_a_max_rows_per_eeg,
                rule=CFG.run_a_dedup_rule,
            )
        stage2_train_rows_after = int(len(stage2_train_df))
        print(
            f'Stage2 train rows: {stage2_train_rows_before} -> {stage2_train_rows_after} '
            f'(dedup={getattr(CFG, "run_a_enable_dedup", False)})'
        )

        if len(stage2_train_df) == 0:
            raise ValueError('Two-stage confident subset is empty. Lower stage2_conf_threshold.')

        stage2_resume_path = stage1_best_ckpt_path if stage1_best_ckpt_path.exists() else stage1_ckpt_path
        print('Stage2 resume checkpoint:', stage2_resume_path)

        stage2_train_loader, stage2_valid_loader = build_dataloaders(stage2_train_df, valid_fold_df, CFG)
        stage2_history = run_training(
            train_loader=stage2_train_loader,
            valid_loader=stage2_valid_loader,
            cfg=CFG,
            baseline_kl=full_baseline_kl,
            epochs=CFG.stage2_extra_epochs,
            checkpoint_name=stage2_ckpt,
            resume=True,
            resume_from=stage2_resume_path,
            weights_only=True,
            reset_tracking=True,
            override_lr=CFG.stage2_lr,
            early_stopping_patience=CFG.stage2_patience,
            stage_name='stage2',
            stage_metadata={
                'stage_label': 'confident_finetune',
                'train_rows': stage2_train_rows_after,
                'train_rows_before_dedup': stage2_train_rows_before,
                'train_rows_after_dedup': stage2_train_rows_after,
                'valid_rows': int(len(valid_fold_df)),
                'confident_threshold': float(CFG.stage2_conf_threshold),
                'resume_from_best': bool(stage1_best_ckpt_path.exists()),
                'tracking_reset': True,
                'dedup_enabled': bool(getattr(CFG, 'run_a_enable_dedup', False)),
                'dedup_max_rows_per_eeg': int(getattr(CFG, 'run_a_max_rows_per_eeg', 2)),
                'dedup_rule': str(getattr(CFG, 'run_a_dedup_rule', 'top_total_votes')),
                'vote_weighting_enabled': bool(getattr(CFG, 'run_a_use_vote_weighting', False)),
                'vote_weighting_mode': str(getattr(CFG, 'run_a_vote_weight_mode', 'sqrt_norm')),
            },
        )

    # --- Final model selection: best of stage1 vs stage2 ---
    import json as _json

    stage1_best_kl = min(full_history['valid_kl']) if full_history else float('inf')
    stage2_best_kl = min(stage2_history['valid_kl']) if stage2_history else float('inf')

    if stage2_best_kl < stage1_best_kl:
        final_stage = 'stage2'
        final_kl = stage2_best_kl
        stage2_best_ckpt_path = CFG.MODELS_DIR / (Path(stage2_ckpt).stem + '_best.pt')
        final_ckpt = str(stage2_best_ckpt_path)
    else:
        final_stage = 'stage1'
        final_kl = stage1_best_kl
        final_ckpt = str(stage1_best_ckpt_path)

    selection = {
        'chosen_stage': final_stage,
        'chosen_kl': float(final_kl),
        'chosen_checkpoint': final_ckpt,
        'stage1_best_kl': float(stage1_best_kl),
        'stage2_best_kl': float(stage2_best_kl),
    }
    sel_dir = CFG.RESULTS_DIR / Path(stage1_ckpt).stem
    sel_dir.mkdir(parents=True, exist_ok=True)
    sel_path = sel_dir / 'final_model_selection.json'
    sel_path.write_text(_json.dumps(selection, indent=2))
    print(f'Final model: {final_stage} (KL={final_kl:.5f}) -> {final_ckpt}')


In [ ]:
# Ensemble export: run the t12 checkpoint on confident_test.csv and save per-sample predictions.
from pathlib import Path

confident_csv_path = CFG.BASE_PATH / 'confident_test.csv'
if not confident_csv_path.exists():
    raise FileNotFoundError(f'Confident test CSV not found: {confident_csv_path}')

ckpt_path = CFG.MODELS_DIR / f'best_conv1d_bigru_fold{CFG.fold}_full_curriculum_t12_dedup4_stage1_best.pt'
if not ckpt_path.exists():
    raise FileNotFoundError(f't12 checkpoint not found: {ckpt_path}')

run_cfg_path = CFG.RESULTS_DIR / 'best_conv1d_bigru_fold0_full_curriculum_t12_dedup4_stage1' / 'run_config_stage1.json'
if not run_cfg_path.exists():
    raise FileNotFoundError(f't12 run_config_stage1.json not found: {run_cfg_path}')

confident_eval_df = pd.read_csv(confident_csv_path).copy()
required_cols = ['eeg_id', *CFG.vote_cols]
missing_cols = [c for c in required_cols if c not in confident_eval_df.columns]
if missing_cols:
    raise ValueError(f'confident_test.csv missing required columns: {missing_cols}')

confident_eval_df['eeg_path'] = confident_eval_df['eeg_id'].apply(
    lambda x: str(CFG.BASE_PATH / 'train_eegs' / f'{x}.parquet')
)

test_votes = confident_eval_df[CFG.vote_cols].values.astype(np.float32)
test_vote_sums = np.clip(test_votes.sum(axis=1, keepdims=True), 1.0, None)
test_soft_labels = test_votes / test_vote_sums
confident_eval_df['soft_labels'] = list(test_soft_labels)

with open(run_cfg_path, 'r', encoding='utf-8') as f:
    t12_payload = json.load(f)

t12_arch = t12_payload.get('model_architecture', {}) or {}

def _normalize_state_dict_keys(state_dict):
    if any(k.startswith('_orig_mod.') for k in state_dict):
        return {k.replace('_orig_mod.', '', 1): v for k, v in state_dict.items()}
    return state_dict

class _T12InferCFG:
    pass

t12_cfg = _T12InferCFG()
for _name in dir(CFG):
    if _name.startswith('_'):
        continue
    setattr(t12_cfg, _name, getattr(CFG, _name))

for _name in ['conv_channels', 'conv_kernels', 'conv_strides', 'gru_hidden', 'gru_layers', 'dropout']:
    if _name in t12_arch:
        setattr(t12_cfg, _name, t12_arch[_name])

# Force the original t12 single-scale Conv1D architecture regardless of the current notebook CFG.
t12_cfg.use_multiscale_conv = False
t12_cfg.multiscale_kernels = (3, 15, 31)

t12_cfg.CACHE_DIR = CFG.CACHE_DIR / 'confident_t12_export'
ensure_dirs([t12_cfg.CACHE_DIR])

export_cache_files = prepare_cache(
    confident_eval_df,
    split='test',
    cfg=t12_cfg,
    force_rebuild=False,
)

export_ds = NPZEEGDataset(export_cache_files, split='test', augment=False)
export_loader = DataLoader(
    export_ds,
    batch_size=min(CFG.batch_size, max(1, len(export_ds))),
    shuffle=False,
    num_workers=CFG.num_workers,
    pin_memory=CFG.pin_memory,
    drop_last=False,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ConvBiGRUAttention(t12_cfg).to(device)
ckpt = torch.load(ckpt_path, map_location=device)
state_dict = ckpt['model_state_dict'] if isinstance(ckpt, dict) and 'model_state_dict' in ckpt else ckpt
state_dict = _normalize_state_dict_keys(state_dict)
model.load_state_dict(state_dict)

preds = predict(model, export_loader, device, t12_cfg)
if len(preds) != len(confident_eval_df):
    raise ValueError(f'Prediction length mismatch: preds={len(preds)} rows={len(confident_eval_df)}')

per_sample_kl = kl_divergence_np(test_soft_labels, preds)
holdout_kl = float(per_sample_kl.mean())

pred_df = pd.DataFrame({'eeg_id': export_ds.eeg_ids})
if 'label_id' in confident_eval_df.columns:
    pred_df.insert(0, 'label_id', confident_eval_df['label_id'].values)
pred_df['per_sample_kl'] = per_sample_kl
pred_df[CFG.vote_cols] = preds

out_path = CFG.RESULTS_DIR / 'confident_test_predictions_t12_ensemble.csv'
pred_df.to_csv(out_path, index=False)

print('t12 checkpoint:', ckpt_path)
print('Rows exported:', len(pred_df))
print('t12 confident_test KL:', holdout_kl)
print('Saved ensemble CSV to:', out_path)
pred_df.head()


In [ ]:
# Confusion matrix for the t12 confident_test export.
# Since HMS targets are soft labels, this uses argmax(target) vs argmax(prediction)
# as a hard-label summary for error analysis.
cm_csv_path = CFG.RESULTS_DIR / 'confident_test_predictions_t12_ensemble.csv'
if not cm_csv_path.exists():
    raise FileNotFoundError(f'Ensemble export CSV not found: {cm_csv_path}. Run the cell above first.')

cm_df = pd.read_csv(cm_csv_path).copy()
required_cols = ['eeg_id', 'per_sample_kl', *CFG.vote_cols]
missing_cols = [c for c in required_cols if c not in cm_df.columns]
if missing_cols:
    raise ValueError(f'Export CSV missing required columns: {missing_cols}')

confident_csv_path = CFG.BASE_PATH / 'confident_test.csv'
truth_df = pd.read_csv(confident_csv_path).copy()
truth_cols = ['eeg_id', *CFG.vote_cols]
if 'label_id' in truth_df.columns:
    truth_cols = ['label_id', *truth_cols]
truth_df = truth_df[truth_cols].copy()

merge_keys = ['eeg_id']
if 'label_id' in cm_df.columns and 'label_id' in truth_df.columns:
    merge_keys = ['label_id', 'eeg_id']

cm_merged = cm_df.merge(truth_df, on=merge_keys, suffixes=('_pred', '_true'), how='inner')
if len(cm_merged) != len(cm_df):
    raise ValueError(f'Confusion-matrix merge mismatch: merged={len(cm_merged)} export={len(cm_df)}')

pred_cols = [f'{c}_pred' for c in CFG.vote_cols]
true_cols = [f'{c}_true' for c in CFG.vote_cols]

y_true = cm_merged[true_cols].values.argmax(axis=1)
y_pred = cm_merged[pred_cols].values.argmax(axis=1)
class_labels = list(CFG.class_names)
num_classes = len(class_labels)

cm_counts = np.zeros((num_classes, num_classes), dtype=np.int64)
for t, p in zip(y_true, y_pred):
    cm_counts[int(t), int(p)] += 1

row_sums = cm_counts.sum(axis=1, keepdims=True)
cm_norm = cm_counts / np.clip(row_sums, 1, None)

cm_counts_df = pd.DataFrame(cm_counts, index=class_labels, columns=class_labels)
cm_norm_df = pd.DataFrame(cm_norm, index=class_labels, columns=class_labels)

counts_out = CFG.RESULTS_DIR / 'confusion_matrix_t12_counts.csv'
norm_out = CFG.RESULTS_DIR / 'confusion_matrix_t12_row_normalized.csv'
fig_out = CFG.RESULTS_DIR / 'confusion_matrix_t12.png'
cm_counts_df.to_csv(counts_out)
cm_norm_df.to_csv(norm_out)

fig, ax = plt.subplots(figsize=(7.5, 6.0))
im = ax.imshow(cm_norm, cmap='Blues', vmin=0.0, vmax=1.0)
ax.set_title('t12 Confusion Matrix on confident_test.csv (argmax target vs argmax prediction)')
ax.set_xlabel('Predicted class')
ax.set_ylabel('True class')
ax.set_xticks(np.arange(num_classes))
ax.set_yticks(np.arange(num_classes))
ax.set_xticklabels(class_labels, rotation=45, ha='right')
ax.set_yticklabels(class_labels)

for i in range(num_classes):
    for j in range(num_classes):
        text = f'{cm_counts[i, j]} ({cm_norm[i, j]:.2f})'
        ax.text(j, i, text, ha='center', va='center', fontsize=8,
                color='white' if cm_norm[i, j] > 0.5 else 'black')

fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='Row-normalized fraction')
fig.tight_layout()
fig.savefig(fig_out, dpi=200, bbox_inches='tight')
plt.show()
plt.close(fig)

print('Saved confusion matrix counts to:', counts_out)
print('Saved row-normalized confusion matrix to:', norm_out)
print('Saved confusion matrix figure to:', fig_out)
cm_counts_df


In [ ]:
"""Generate dedup ablation curve: cap_rows_per_eeg (n) vs best val KL divergence.
Uses brokenaxes to show a // break between the data range and the global-prior baseline.
"""
from pathlib import Path
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from brokenaxes import brokenaxes

OUT_PATH = Path(
    '/home/littl/ECE247A_Final_Project/JZ/results'
    '/best_conv1d_bigru_fold0_full_curriculum_t12_dedup4_stage1'
    '/dedup_ablation_curve.png'
)
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# ── data ─────────────────────────────────────────────────────────────────────
ns      = [2,      3,      4,      5     ]
val_kls = [0.79321, 0.77106, 0.74585, 0.74726]

PRIOR_KL = 1.3771
T3_KL    = 0.776

# ── figure ────────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(5.5, 2.8))
bax = brokenaxes(
    ylims=((0.70, 0.81), (1.34, 1.44)),
    hspace=0.12,
    fig=fig,
)

# Main curve
bax.plot(ns, val_kls, 'o-', color='#2471A3', linewidth=1.8, markersize=6,
         markerfacecolor='white', markeredgewidth=1.8, zorder=3,
         label='Best val KL (this work)')

# Best point
best_n  = ns[val_kls.index(min(val_kls))]
best_kl = min(val_kls)
bax.scatter([best_n], [best_kl], s=80, color='#2471A3', zorder=4)

# Reference lines
bax.axhline(PRIOR_KL, color='#C0392B', linewidth=1.2, linestyle='--', zorder=2,
            label='Mean-label baseline')
bax.axhline(T3_KL, color='#7D6608', linewidth=1.2, linestyle=':', zorder=2,
            label='No dedup')

# Improvement shading (bottom panel only — below T3_KL)
bax.axs[0].axhspan(0.70, T3_KL, alpha=0.05, color='green', zorder=0)

# ── axes formatting ───────────────────────────────────────────────────────────
bax.set_xlabel('cap_rows_per_eeg  (n)', fontsize=9, labelpad=18)
bax.set_ylabel('Best validation KL divergence', fontsize=9, labelpad=30)

for ax in bax.axs:
    ax.set_xticks(ns)
    ax.tick_params(labelsize=8)
    ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.2f'))
    ax.grid(axis='y', linestyle=':', linewidth=0.6, color='#cccccc', zorder=0)
    ax.spines['right'].set_visible(False)

bax.axs[0].legend(fontsize=7.5, loc='upper right', framealpha=0.8)

fig.suptitle('Effect of training-set deduplication on validation KL',
             fontsize=9.5, fontweight='bold')

fig.savefig(OUT_PATH, dpi=200, bbox_inches='tight', facecolor='white')
print(f'Saved: {OUT_PATH}')
plt.close(fig)
